## Distributed System Health Prediction - Data Preprocessing Pipeline
```
Author: Will Kusper

Date: 1/17/2026

Goals:
    1. Handle categorical features
    2. Create train/test split
    3. Engineer new features
    4. Scale features where needed
```

In [19]:
import pandas as pd
import numpy as np
import 

# Load the dataset
df = pd.read_csv('../data/distributed_system_architecture_stress_dataset.csv')

SyntaxError: invalid syntax (3820852385.py, line 3)

# Handle categorical features

In [ ]:
# One-hot encode the categorical columns (Except for 'system_state')
architecture_dummies = pd.get_dummies(df['architecture_type'], prefix='architecture_type', dtype=int)
deployment_dummies = pd.get_dummies(df['deployment_type'], prefix='deployment_type', dtype=int)
communication_dummies = pd.get_dummies(df['communication_type'], prefix='communication_type', dtype=int)
root_cause_dummies = pd.get_dummies(df['root_cause'], prefix='root_cause', dtype=int)

# Replace the original columns with the encoded columns
df = df.drop(columns=['architecture_type', 'deployment_type', 'communication_type', 'root_cause'])
df = pd.concat([df, architecture_dummies, deployment_dummies, communication_dummies, root_cause_dummies], axis=1)

# Save the updated dataframe to CSV
df.to_csv('../data/distributed_system_architecture_stress_dataset_encoded.csv', index=False)
print('Dataframe saved to CSV with one-hot encoding applied to categorical columns.')

Dataframe saved to CSV with one-hot encoding applied to categorical columns.


## Why I chose this approach
One-hot encoding is good here since there is no inherent order within the columns that are encoded. Thus, there shouldn't be a higher or lower value assigned based off of one category. This is good because most ML algorithms perform mathematical operations. For example, if I used label encoding, I would assign something like :
    - monolith = 0
    - microservices = 1
    - serverless = 2
    - event_driven = 3
    - hybrid = 4
Then, a model might think something like "microservices are close to monolith", or that "serverless is the average-case architecture_type", or other thoughts. Because architectural choices should be independent, this would create a false mathematical relationship between them. 

## How One-Hot fixes this
By creating a separate binary column, where 
    architecture_type = monolith : [1, 0, 0, 0, 0]
    architecture_type = microservices : [0, 1, 0, 0, 0]

Each architecture is now independent. The model will be able to learn how each one individually affects system health. This prevents the model from learning nonexistent patterns, and instead to discover the real relationships between columns.

# Feature Engineering

In [22]:
# load_intensity : avg_payload_kb * requests_per_second
# shows how much information the system is handling per second
df['load_intensity'] = df['avg_payload_kb'] * df['requests_per_second']

# peak_load_intensity : load_intensity * peak_traffic_multiplier
# estimates the maximum load the system experiences during peak times
df['peak_load_intensity'] = df['load_intensity'] * df['peak_traffic_multiplier']

# requests_per_service : requests_per_second / num_services
# indicates the average number of requests each service handles per second
df['requests_per_service'] = df['requests_per_second'] / df['num_services']

# database_service_density : num_services / num_databases
# reflects how services are distributed in relation to databases
df['database_service_density'] = df['num_services'] / (df['num_databases'] + 1)  # +1 to avoid division by zero
# writes_per_second : requests_per_second * read_write_ratio
# estimates the number of write operations per second
df['writes_per_second'] = df['requests_per_second'] * df['read_write_ratio']

# error_density : error_rate_percent / requests_per_second
# measures the frequency of errors relative to the number of requests
df['error_density'] = df['error_rate_percent'] / df['requests_per_second']

# Save the updated dataframe with new features to CSV
df.to_csv('../data/distributed_system_architecture_stress_dataset_enhanced.csv', index=False)

## Why each feature was added

`load_intensity` : This feature is a combination of two features, avg_payload_kb and requests_per_second, to better capture the load is experiencing. For example, two servers may experience an opposite pair of values for those two features. The load_intensity feature will show that servers in similar scenarios are actually experiencing the same load.

`peak_load_intensity` : Similarly, this feature shows the load a server faces in peak traffic. It is effective for the same reasons as above

`requests_per_service` : Services that face heavier loads are more likely to fail. Services can only handle so many requests before failure. In addition, each service may be competing for resources. If there are a higher number of requests per service, this may mean more competition for resources and thus more failures. 

`database_service_density` : This feature reflects how many services are deployed in relation to how many databases are utilized. A higher density means competition in each conneciton pool. More services per database means more concurrent transactions as well, which increases the probability that there are lock conflicts, or blocked transactions. Lastly, more database queries increases request timeouts as wait times increase for those queries.

`writes_per_second` : Writes are an expensive operation, and tracking the amount of writes shows that cost. Writes consume a lot of CPU and memory, which slows down systems. Slower systems mean more waiting. More waiting leads to more timeouts, which leads to retries and thus exponential load.

`error_density` : Raw error rate hides how much a system is actually suffering. A system that experiences high failure under very high load is much different than one that experiences high failure under a smaller load. Dividing by request volume normalizes this. 

# Create Train/Test splits

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target variable
X = df.drop(columns=['system_state'])
y = df['system_state']

# Create stratified train/test split (80/20)
# stratify=y ensures each split has the same proportion of each class
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f'Training set size: {len(X_train)} samples')
print(f'Test set size: {len(X_test)} samples')
print('\nClass distribution in training set:')
print(y_train.value_counts(normalize=True))
print('\nClass distribution in test set:')
print(y_test.value_counts(normalize=True))
print('\nOriginal class distribution:')
print(y.value_counts(normalize=True))